In [ ]:
!pip install -q langchain_community
!pip install -q replicate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.0 MB/s eta 0:00:00


In [ ]:
from langchain_community.llms import Replicate
import os
from google.colab import userdata
# Set the API token
api_token = userdata.get('api_token')
os.environ["REPLICATE_API_TOKEN"] = api_token
# Model setup
model = "ibm-granite/granite-3.2-8b-instruct"
output = Replicate(
model=model,
replicate_api_token=api_token,
)

In [ ]:
# Define the customer reviews
customer_reviews = [
"Absolutely loved this place! The atmosphere is warm and welcoming, with a true Italian vibe. The pasta was perfectly cooked, full of flavor, and the pizza had the most delicious crispy crust. The staff were very attentive and friendly, making the whole experience even better. Highly recommend this restaurant to anyone craving authentic Italian food in town!.",
"Service was great, The young men and women were friendly and genuine interacting with us. I liked that, being my first meal in Melbourne. The food itself was average. Had a pizza and some ribs or chicken wings with fries. Neither was shout worthy. I see a lot of students in there; it is not particularly cheap, nor expensive most food places have their dishes priced in the $20-30 range.",
"The food was highly talked about and was clearly a local favorite. I personally found the flavors to not have a broad range nor to entice the senses. It was fine, but nothing awe-inspiring. I ate it all very happily. Now, the ambiance was really nice. They made good use of the space and really made the environment welcoming. Speaking of welcoming, the staff were great! Very kind, welcoming, and enjoyed banter and getting into the stories of the guests. All star waitstaff."
"I’ve tried a few pastas and the pesto chicken and Paul’s special are my favourites, but the pizza is better. The pesto pasta is alright. The shared platter is a good order if you have minimum 3 people."
"Majestic food and awesome service at this beautiful restaurant. Our family has thoroughly enjoyed this place for last 1 years and the food is always good and my children are in awe of it. Thank you so much."
]
# Refine the prompt to include reviews
reviews_text = "\n".join([f"Review {i+1}: {review}" for i, review
in enumerate(customer_reviews)])
prompt = f"""
Classify all reviews into three categories Positive, Negative, or Mixed:
{reviews_text}
"""
# Invoke the model with the example prompt
response = output.invoke(prompt)
# Print the response
print("Granite Model Response:\n")
print(response)

Granite Model Response:

Review 1: Positive
Review 2: Mixed
Review 3: Positive


In [ ]:
# Define refined prompt
refined_prompt = f"""
Classify these reviews as positive, negative, or mixed, and tag
relevant categories (delicious, service, or
atmosphere):
{reviews_text}
"""
# Invoke the model with the example prompt
response = output.invoke(refined_prompt)
# Print the response
print("Granite Model Refined Response:\n")
print(response)

Granite Model Refined Response:

Review 1:
- Classification: Positive
- Categories:
  - Delicious: High praise for pasta and pizza, mention of perfect cooking and flavorful food.
  - Service: Positive feedback on staff being attentive and friendly.
  - Atmosphere: Warm and welcoming Italian vibe.

Review 2:
- Classification: Mixed
- Categories:
  - Service: Positive, describing staff as friendly and genuine.
  - Atmosphere: No specific comments, assumed neutral.
  - Delicious: Food described as average, with no standout qualities mentioned.

Review 3:
- Classification: Mixed
- Categories:
  - Delicious: Mixed feelings about the food, describing flavors as not broad or enticing, but still fine and happily eaten.
  - Service: Highly positive, describing staff as kind, welcoming, and enjoyable to interact with.
  - Atmosphere: Positive, praising the use of space and welcoming environment.


In [ ]:
# Define the prompt to complete the task in 2 steps
multitask_prompt = f"""
Complete the task in 2 steps.
Step 1: Classify these reviews as positive, negative, or mixed.
Step 2: For each review, identify relevant categories: delicious, service, or atmosphere.
{reviews_text}
"""
response = output.invoke(multitask_prompt)
print("Granite Model Response:\n")
print(response)

Granite Model Response:

Step 1: Classification

Review 1: Positive
Review 2: Mixed
Review 3: Positive

Step 2: Categorization

Review 1:
- Delicious: High praise for the pasta and pizza, particularly the crust.
- Service: Highly commends the staff for being attentive and friendly.
- Atmosphere: Describes a warm and welcoming Italian vibe.

Review 2:
- Delicious: Food is described as "average", with neither pizza nor ribs/chicken wings being "shout-worthy".
- Service: Praises the service as "great", with staff being friendly and genuine.
- Atmosphere: No specific mention, but implies a casual setting given the student presence.

Review 3:
- Delicious: While the food is described as "fine", certain dishes like the pesto chicken and Paul's special are highlighted as favorites.
- Service: High praise for the staff, described as "star waitstaff", kind, welcoming, and enjoying banter with guests.
- Atmosphere: Describes the ambiance as "really nice", making good use of the space and being w

In [ ]:
# Define the example to guide the model
formatted_prompt = f"""
Classify these reviews as Positive, Negative, or Mixed, and tag
relevant categories. Use this format:
- Sentiment: [Sentiment]
- Categories: [Categories].

{reviews_text}
"""
# Invoke the model with prompt
response = output.invoke(formatted_prompt)
# Print the response
print("Granite Model Formatted Response:\n")
print(response)

Granite Model Formatted Response:

- Sentiment: Positive
  - Categories: Atmosphere, Staff, Food Quality (Pasta, Pesto Chicken, Paul's Special), Shared Platter

- Sentiment: Mixed
  - Categories: Service, Food Quality (Pizza, Ribs/Chicken Wings, Fries), Pricing

- Sentiment: Positive
  - Categories: Atmosphere, Staff, Food Quality (Pesto Pasta, Shared Platter), Family-friendly, Long-term Enjoyment (1 year), Children's Menu (implied)


In [ ]:
# Set model parameters for prompting with default values
parameters = {
"top_k": 0,
"top_p": 1.0,
"max_tokens": 256,
"min_tokens": 0,
"random_seed": None,
"repetition_penalty": 1.0,
"stopping_criteria": "length (256 tokens)",
"stopping_sequence": None
}

In [ ]:
# Add initial prompt
refined_prompt = f""":
Classify these reviews as positive, negative, or mixed, and tag
relevant focus areas such as food quality, service, or
atmosphere
{reviews_text}
"""
# Invoke the model
response = output.invoke(refined_prompt, parameters=parameters)
# Print the response
print("Granite Model Refined Response:\n")
print(response)

Granite Model Refined Response:

Review 1: Positive
- Focus areas: Atmosphere, Food quality (pasta, pizza)

Review 2: Mixed
- Focus areas: Service, Food quality (pizza, ribs/chicken wings)

Review 3: Positive
- Focus areas: Atmosphere, Service, Food quality (pasta, pesto chicken, Paul's special, pizza), Overall experience (family enjoyment)
